# Avance 5 — Ensambles para la regresión de potencia

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [04b](04b_Regresion_actividad.ipynb) y del [04c](04c_Regresion_actividad_huellas.ipynb) para la fase 2. Evalúa modelos de **ensamble** (homogéneos y heterogéneos) y de **tiempo de falla acelerado (AFT)** para la predicción continua de potencia biológica (`pActividad`). Los ensambles reciben la representación combinada de descriptores físico-químicos y huellas moleculares **Sort & Slice** aprendidas dentro de cada pliegue.

Para los ensambles heterogéneos (`votacion` y `apilado`), las huellas se integran dentro de cada estimador base, asegurando que el vocabulario de subestructuras se aprenda en su pliegue interno correspondiente. El modelo `ApiladoAgrupado` implementa una meta-regresión con validación cruzada interna agrupada por andamio molecular (`GroupKFold`), evitando la fuga de información entre análogos estructurales. Por su parte, `RegresionAFT` evalúa el enfoque de supervivencia paramétrica con XGBoost sobre las mediciones censuradas sin descartar intervalos.

## 1. Preparación

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por proceso: evita que los procesos compitan por los hilos de BLAS con n_jobs=-1

import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.features.huellas import AgregarHuellas, agrupar_iguales, identificadores_morgan
from src.models.aft import RegresionAFT
from src.models.baseline_regresion import SoloExactas, evaluar_baselines, tabla
from src.models.ensambles import AfinadoAgrupado, ApiladoAgrupado, ensambles
from src.models.pisos import MediaPorDiana
from src.tracking import Corrida

# Rejilla para el afinado de kNN (procedente del params.yaml de la corrida 2026-10-26-antonio-2.9-baseline-regresion-sencilla-exactas)
REJILLA_KNN = {
    "kneighborsregressor__n_neighbors": [15, 30],
    "kneighborsregressor__weights": ["uniform", "distance"],
}

# Semillas para la evaluación (sondeo con una semilla; corrida final con tres)
SEMILLAS = (0, 1, 2)
SUFIJO = "-sondeo" if len(SEMILLAS) == 1 else ""
BITACORA = RAIZ / "_parcial_05b.log"      # ignorado por *.log; seguro y progreso en vivo

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")


### Las huellas y el código que las arma

Una huella **Sort & Slice** es una tabla de 0 y 1: una columna por subestructura (el entorno de radio 2 alrededor de un átomo, el mismo que ECFP4) y un 1 si la molécula la tiene. De todas las subestructuras que aparecen se conservan las **2,048 más frecuentes**. A diferencia de una huella plegada por hash, cada columna es **una** subestructura, y por eso una columna con peso alto se puede traducir a un fragmento concreto.

**El vocabulario se aprende en cada pliegue, con sus filas de entrenamiento.** Es lo que más importa de este cuaderno. Qué subestructuras son «las más frecuentes» depende de con qué filas se cuenten: si se cuentan con todas, la validación ya influyó en qué columnas existen, y eso es una fuga que no aparece en ninguna métrica y hace que las huellas parezcan mejores de lo que son. Por eso las moléculas se leen una sola vez (una columna por cada subestructura que aparece en alguna) y la elección de las 2,048 ocurre dentro de la canalización, en cada ajuste.

Dos diferencias con el `SortAndSlice` de `src/features/molecular.py`, que cuenta por molécula sobre una lista fija: aquí la frecuencia se cuenta por fila (en la fase 2 un compuesto aparece en 1.18 pares en promedio) y la elección se repite en cada ajuste. `tests/test_huellas.py` comprueba que, cuando cada molécula aparece una vez, el vocabulario y la matriz son idénticos a los de `SortAndSlice`.

**El piso por diana.** El 04b solo tiene un piso, predecir siempre el promedio de todas las mediciones. Pero el corte de 10 µM no significa lo mismo en cada diana (deja a `LRRK2` en 98.1% de activos y a `NFE2L2` en 50.7%), y un modelo que solo sepa contra qué diana se midió ya supera ese promedio sin entender química. `MediaPorDiana` predice la potencia media de la diana de cada par con las filas de entrenamiento del pliegue, sin mirar los descriptores, y revienta si las columnas que se le pasan no son un one-hot.

### Los datos

Son los mismos pares, con el mismo criterio, que el 04b: entrenamiento y validación (la prueba reservada no se toca en este sondeo de validación cruzada), sin los pares que no informan nada, y con la diana como columnas de 0 y 1. Los modelos se entrenan con las mediciones exactas y la validación incluye las censuradas.

**Nota metodológica sobre la partición de prueba:** Para la corrida final con evaluación en prueba, `X_mixta` debe construirse sobre todas las filas combinadas usando una máscara booleana para separarlas. Esto asegura que el espacio de columnas de huellas (`identificadores_morgan`) se defina una sola vez de forma unívoca, garantizando que las columnas representen exactamente los mismos fragmentos en entrenamiento y prueba.

In [ ]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares.merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = ~(np.isneginf(inferior) & np.isposinf(superior)) & (inferior <= superior)
print(f"{len(d):,} pares en total · {(~util).sum():,} sin información y salen")

d = d[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
es_prueba = (d["particion"] == "prueba").to_numpy()
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
exacta = inferior == superior
y_cls = d["activo_10um"].to_numpy(float)       # la clase derivada a 10 µM; NaN si no decide
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()          # solo_chembl, solo_npass, mixto

assert (~es_prueba).sum() == 56_241, "las filas de ajuste cambiaron respecto al 04c"
assert exacta[~es_prueba].sum() == 46_100, "las exactas de ajuste cambiaron"

print(f"{len(d):,} pares ({es_prueba.sum():,} de prueba; {(~es_prueba).sum():,} de ajuste) · "
      f"{exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas")


### Las huellas de los pares

Se leen los SMILES una sola vez y se arma una matriz dispersa con una columna por subestructura. **Todavía no se elige ningún vocabulario**: eso pasa dentro de cada pliegue. Antes se agrupan las subestructuras que valen lo mismo en todos los pares (porque siempre aparecen juntas, como el `S` y el `O=S` de un sulfonilo): se queda una sola columna y se guarda, en `equivalentes`, el nombre de todas las que representa. Las columnas de descriptores y de diana van primero y las huellas después, en una sola matriz dispersa sin densificar.

In [ ]:
t0 = time.perf_counter()
huellas, identificadores = identificadores_morgan(d["smiles"])
n_distintas = huellas.shape[1]
huellas, identificadores, equivalentes = agrupar_iguales(huellas, identificadores)
n_agrupadas = sum(len(e) > 1 for e in equivalentes)
N_DESC = len(cols)                       # descriptores
N_DENSAS = X.shape[1]                    # descriptores + diana
DIANA = tuple(range(N_DESC, N_DENSAS))
N_BITS = 2048                            # tamaño del vocabulario de Sort & Slice
X_mixta = sparse.hstack([sparse.csr_matrix(X), huellas]).tocsr()
print(f"{n_distintas:,} subestructuras distintas en {huellas.shape[0]:,} pares; "
      f"{n_distintas - huellas.shape[1]:,} valían lo mismo que otra y quedan {huellas.shape[1]:,} columnas "
      f"({n_agrupadas} de ellas representan a más de una) · {time.perf_counter() - t0:.1f} s")

# Separación con la máscara booleana: ajuste (entrenamiento + validación) vs prueba.
# Las huellas se armaron sobre el conjunto completo para que las columnas
# tengan exactamente el mismo significado en ajuste y prueba.
X_mixta_prueba, X_prueba = X_mixta[es_prueba], X[es_prueba]
inferior_prueba, superior_prueba = inferior[es_prueba], superior[es_prueba]

X_mixta = X_mixta[~es_prueba]
X = X[~es_prueba]
inferior = inferior[~es_prueba]
superior = superior[~es_prueba]
g = g[~es_prueba]
procedencia = procedencia[~es_prueba]
y_cls = y_cls[~es_prueba]
exacta = exacta[~es_prueba]


## 2. Ensambles de regresión

Se configuran cinco familias de ensamble combinando descriptores moleculares y huellas estructurales:
- **Ensambles homogéneos:** `bosque` (`RandomForestRegressor`), `boosting` (`HistGradientBoostingRegressor`) y `bagging_knn` (`BaggingRegressor` sobre kNN).
- **Ensambles heterogéneos:** `votacion` (`VotingRegressor`) y `apilado` (`ApiladoAgrupado`) con estimadores base Ridge y kNN.
- **Regresión AFT censurada:** `RegresionAFT` (árboles potenciados con XGBoost bajo verosimilitud de supervivencia).
- **Afinado con partición agrupada:** `AfinadoAgrupado` realiza la búsqueda de hiperparámetros con validación cruzada interna agrupada por andamio (`GroupKFold`), evitando la sobreestimación por fuga de información entre análogos moleculares al igual que en el apilado.

Para garantizar que tanto `ApiladoAgrupado` como `AfinadoAgrupado` reciban los grupos del andamio y que el preprocesamiento de huellas se ajuste honestamente sin fuga en pliegues internos, `AgregarHuellas` se coloca dentro de las bases en lugar de envolver al ensamble heterogéneo.

A continuación se muestra el código implementado en `src/models/ensambles.py` y `src/models/aft.py`:

```python
class ApiladoAgrupado(BaseEstimator, RegressorMixin):
    """Ensamble heterogéneo apilado (stacking) con validación cruzada agrupada.

    Envuelve `StackingRegressor` garantizando que las predicciones fuera de pliegue
    empleadas para entrenar el meta-estimador respeten la partición por andamios
    estructurales (`GroupKFold`).

    Declara `necesita_grupos = True` para que los arneses de evaluación externa
    (`cv.evaluar` o `evaluar_baselines`) le suministren los grupos del pliegue.
    """

    necesita_grupos: bool = True

    def __init__(
        self,
        bases: Sequence[tuple[str, Any]] | Mapping[str, Any],
        final: Any = None,
        *,
        pliegues: int = 3,
        semilla: int = 0,
        n_jobs: int | None = None,
        passthrough: bool = False,
    ) -> None:
        self.bases = bases
        self.final = final
        self.pliegues = pliegues
        self.semilla = semilla
        self.n_jobs = n_jobs
        self.passthrough = passthrough

    def _obtener_lista_bases(self) -> list[tuple[str, Any]]:
        """Normaliza la colección de estimadores base a una lista de tuplas (nombre, estimador)."""
        if isinstance(self.bases, Mapping):
            return list(self.bases.items())
        return list(self.bases)

    def fit(
        self,
        X: Any,
        y: Any,
        grupos: Any = None,
        sample_weight: Any = None,
    ) -> ApiladoAgrupado:
        """Ajusta el ensamble apilado respetando los grupos en la partición interna."""
        if grupos is None:
            raise ValueError(
                "ApiladoAgrupado requiere 'grupos' para la validación cruzada interna "
                "por andamio (scaffold) para evitar fuga de datos entre pliegues."
            )

        grupos_arr = np.asarray(grupos)
        y_arr = np.asarray(y).ravel()

        if len(grupos_arr) != len(y_arr):
            raise ValueError(
                f"La longitud de 'grupos' ({len(grupos_arr)}) debe coincidir con la cantidad "
                f"de muestras en 'y' ({len(y_arr)})."
            )

        if self.pliegues < 2:
            raise ValueError(
                f"El número de pliegues debe ser al menos 2, pero se recibió pliegues={self.pliegues}."
            )

        n_grupos_unicos = len(np.unique(grupos_arr))
        if self.pliegues > n_grupos_unicos:
            raise ValueError(
                f"El número de pliegues (pliegues={self.pliegues}) no puede ser mayor que "
                f"el número de grupos únicos ({n_grupos_unicos})."
            )

        lista_bases = self._obtener_lista_bases()
        if not lista_bases:
            raise ValueError("ApiladoAgrupado requiere al menos un modelo base en 'bases'.")

        for item in lista_bases:
            if (
                not isinstance(item, (tuple, list))
                or len(item) != 2
                or not isinstance(item[0], str)
            ):
                raise ValueError(
                    "Cada elemento de 'bases' debe ser una tupla (nombre, estimador) con un nombre de tipo str."
                )

        cv = GroupKFold(n_splits=self.pliegues, shuffle=True, random_state=self.semilla)
        particiones = list(cv.split(X, y_arr, grupos_arr))
        self.particiones_ = particiones

        estimador = StackingRegressor(
            estimators=lista_bases,
            final_estimator=self.final,
            cv=particiones,
            n_jobs=self.n_jobs,
            passthrough=self.passthrough,
        )
        if sample_weight is not None:
            estimador.fit(X, y_arr, sample_weight=sample_weight)
        else:
            estimador.fit(X, y_arr)

        self.estimador_ = estimador
        self.estimators_ = estimador.estimators_
        self.named_estimators_ = estimador.named_estimators_
        self.final_estimator_ = estimador.final_estimator_
        if hasattr(estimador, "stack_method_"):
            self.stack_method_ = estimador.stack_method_
        if hasattr(estimador, "n_features_in_"):
            self.n_features_in_ = estimador.n_features_in_
        if hasattr(estimador, "feature_names_in_"):
            self.feature_names_in_ = estimador.feature_names_in_
        return self

    def predict(self, X: Any) -> np.ndarray:
        """Delega la predicción al estimador apilado ajustado."""
        check_is_fitted(self, "estimador_")
        return self.estimador_.predict(X)

    def transform(self, X: Any) -> np.ndarray:
        """Retorna las predicciones de los estimadores base sobre X (meta-características)."""
        check_is_fitted(self, "estimador_")
        return self.estimador_.transform(X)

    def get_feature_names_out(self, input_features: Any = None) -> np.ndarray:
        """Obtiene los nombres de las características de salida transformadas."""
        check_is_fitted(self, "estimador_")
        return self.estimador_.get_feature_names_out(input_features)



class AfinadoAgrupado(BaseEstimator, RegressorMixin):
    """Búsqueda de hiperparámetros con validación cruzada interna agrupada por andamio.

    Envuelve `GridSearchCV` asegurando que las particiones internas respeten los
    grupos estructurales (`GroupKFold`), evitando la sobreestimación por fuga de
    información entre análogos moleculares.

    Utiliza `GroupKFold` y no `StratifiedGroupKFold` porque la variable objetivo
    (`pActividad`) es continua en regresión (a diferencia de `BusquedaAnidada` en
    `src/models/ajuste.py`, diseñada para clasificación binaria). Esta clase duplica
    temporalmente a `BusquedaAnidada` para no romper la entrega del Avance 5;
    ambas se fusionarán en un estimador unificado tras la entrega.
    """

    necesita_grupos: bool = True

    def __init__(
        self,
        canalizacion: Any,
        rejilla: Mapping[str, Sequence[Any]] | Any,
        *,
        pliegues: int = 3,
        metrica: str = "neg_root_mean_squared_error",
        semilla: int = 0,
        n_jobs: int | None = None,
    ) -> None:
        self.canalizacion = canalizacion
        self.rejilla = rejilla
        self.pliegues = pliegues
        self.metrica = metrica
        self.semilla = semilla
        self.n_jobs = n_jobs

    def fit(self, X: Any, y: Any, grupos: Any = None) -> AfinadoAgrupado:
        """Ajusta la búsqueda de hiperparámetros respetando los grupos en la partición interna."""
        if grupos is None:
            raise ValueError(
                "AfinadoAgrupado requiere 'grupos' para la validación cruzada interna "
                "por andamio (scaffold) para evitar fuga de datos entre pliegues."
            )

        grupos_arr = np.asarray(grupos)
        y_arr = np.asarray(y).ravel()

        if len(grupos_arr) != len(y_arr):
            raise ValueError(
                f"La longitud de 'grupos' ({len(grupos_arr)}) debe coincidir con la cantidad "
                f"de muestras en 'y' ({len(y_arr)})."
            )

        if self.pliegues < 2:
            raise ValueError(
                f"El número de pliegues debe ser al menos 2, pero se recibió pliegues={self.pliegues}."
            )

        n_grupos_unicos = len(np.unique(grupos_arr))
        if self.pliegues > n_grupos_unicos:
            raise ValueError(
                f"El número de pliegues (pliegues={self.pliegues}) no puede ser mayor que "
                f"el número de grupos únicos ({n_grupos_unicos})."
            )

        particiones = list(
            GroupKFold(n_splits=self.pliegues, shuffle=True, random_state=self.semilla).split(
                X, y_arr, grupos_arr
            )
        )
        self.busqueda_ = GridSearchCV(
            clone(self.canalizacion),
            dict(self.rejilla),
            scoring=self.metrica,
            cv=particiones,
            n_jobs=self.n_jobs,
            refit=True,
        ).fit(X, y_arr)
        self.best_estimator_ = self.busqueda_.best_estimator_
        self.elegidos_ = self.busqueda_.best_params_
        self.puntaje_interno_ = float(self.busqueda_.best_score_)
        return self

    def predict(self, X: Any) -> np.ndarray:
        """Predice utilizando el mejor estimador ajustado."""
        check_is_fitted(self, "busqueda_")
        return self.busqueda_.best_estimator_.predict(X)



def ensambles(
    bases: Sequence[tuple[str, Any]] | Mapping[str, Any] | None = None,
    semilla: int = 0,
    n_jobs: int | None = None,
) -> dict[str, Any]:
    """Catálogo de modelos de ensamble para regresión de potencia.

    Devuelve un diccionario con las cinco familias de ensamble requeridas:
    1. `bosque`: `RandomForestRegressor` (homogéneo, bagging de árboles).
    2. `boosting`: `HistGradientBoostingRegressor` (homogéneo, boosting de gradiente).
    3. `bagging_knn`: `BaggingRegressor` sobre `KNeighborsRegressor` (homogéneo, bagging).
    4. `votacion`: `VotingRegressor` (heterogéneo, promedio simple sin partición interna).
    5. `apilado`: `ApiladoAgrupado` (heterogéneo, apilado con validación interna agrupada).

    Las bases heterogéneas por omisión son `Ridge` y `KNeighborsRegressor`:
    en el #164 kNN obtuvo la mejor concordancia del cuaderno (0.756 con
    descriptores y huellas; 0.739 con descriptores solos) y Ridge representa el
    control lineal regularizado. El MLP queda fuera porque fue el único modelo
    que empeoró (−0.027).
    """
    if bases is None:
        bases_apilado = [
            ("ridge", Ridge()),
            ("knn", KNeighborsRegressor()),
        ]
        bases_votacion = list(bases_apilado)
    elif isinstance(bases, Mapping):
        bases_apilado = bases
        bases_votacion = list(bases.items())
    elif not isinstance(bases, (list, tuple)):
        bases_apilado = list(bases)
        bases_votacion = list(bases_apilado)
    else:
        bases_apilado = bases
        bases_votacion = list(bases)

    return {
        "bosque": RandomForestRegressor(random_state=semilla, n_jobs=n_jobs),
        "boosting": HistGradientBoostingRegressor(random_state=semilla),
        "bagging_knn": BaggingRegressor(
            estimator=KNeighborsRegressor(),
            random_state=semilla,
            n_jobs=n_jobs,
        ),
        "votacion": VotingRegressor(estimators=bases_votacion, n_jobs=n_jobs),
        "apilado": ApiladoAgrupado(
            bases=bases_apilado,
            semilla=semilla,
            n_jobs=n_jobs,
        ),
    }
```

```python
class RegresionAFT(BaseEstimator, RegressorMixin):
    """Regresor no lineal para datos censurados mediante Accelerated Failure Time (XGBoost).

    Envuelve la interfaz de supervivencia de XGBoost adaptándola al contrato de
    `fit(X, inferior, superior)` del proyecto y convirtiendo las predicciones a pActividad.
    """

    def __init__(
        self,
        distribucion: str = "normal",
        escala: float = 1.0,
        rondas: int = 100,
        profundidad: int = 6,
        semilla: int = 0,
        n_jobs: int | None = None,
    ) -> None:
        self.distribucion = distribucion
        self.escala = escala
        self.rondas = rondas
        self.profundidad = profundidad
        self.semilla = semilla
        self.n_jobs = n_jobs

    def fit(
        self,
        X: Any,
        inferior: Any,
        superior: Any,
        grupos: Any = None,
    ) -> RegresionAFT:
        """Ajusta el modelo AFT sobre las mediciones en escala de concentración (µM).

        1. Convierte las cotas de pActividad a micromolar con `limites_en_micromolar`.
        2. Descarta filas sin cota informativa, mal definidas o contradictorias.
        3. Configura `label_lower_bound` y `label_upper_bound` en un `DMatrix`.
        4. Entrena el ensamble de árboles con la pérdida `survival:aft`.
        """
        inf_um, sup_um = limites_en_micromolar(inferior, superior)

        # Descartar lo que no informa o está mal definido:
        # inf_um > 0 y finito elimina intervalos no informativos [0, +inf], ceros e infinitos;
        # sup_um >= inf_um asegura que sup_um no sea NaN (p. ej. cota inferior en pActividad NaN)
        # ni contradictorio (sup_um < inf_um).
        valida = (inf_um > 0) & np.isfinite(inf_um) & (sup_um >= inf_um)
        self.n_descartadas_ = int((~valida).sum())

        if not np.any(valida):
            raise ValueError(
                "No hay filas válidas con cota inferior positiva en µM (inf_um > 0) para ajustar el AFT."
            )

        X_arr = np.asarray(X, dtype=float)[valida]
        inf_valid = inf_um[valida].astype(np.float32)
        sup_valid = sup_um[valida].astype(np.float32)

        self.n_features_in_ = X_arr.shape[1]

        dtrain = xgb.DMatrix(X_arr)
        dtrain.set_float_info("label_lower_bound", inf_valid)
        dtrain.set_float_info("label_upper_bound", sup_valid)

        params: dict[str, Any] = {
            "objective": "survival:aft",
            "eval_metric": "aft-nloglik",
            "aft_loss_distribution": self.distribucion,
            "aft_loss_distribution_scale": self.escala,
            "tree_method": "hist",
            "max_depth": self.profundidad,
            "seed": self.semilla,
        }
        if self.n_jobs is not None:
            params["nthread"] = self.n_jobs

        self.booster_ = xgb.train(params, dtrain, num_boost_round=self.rondas)
        return self

    def predict(self, X: Any) -> np.ndarray:
        """Predice la potencia esperada en escala de pActividad (6 − log₁₀ µM)."""
        check_is_fitted(self, "booster_")

        X_arr = np.asarray(X, dtype=float)
        dtest = xgb.DMatrix(X_arr)
        pred_um = self.booster_.predict(dtest)

        # Asegurar positividad antes del logaritmo
        pred_um_pos = np.maximum(pred_um, 1e-12)
        return 6.0 - np.log10(pred_um_pos)
```

In [ ]:
def con_huellas(m):
    return make_pipeline(AgregarHuellas(N_DENSAS, n_bits=N_BITS), m)

bases = [("ridge", con_huellas(make_pipeline(MinMaxScaler(), Ridge()))),
         ("knn",   con_huellas(make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15, n_jobs=-1))))]
e = ensambles(bases=bases, semilla=SEMILLAS[0], n_jobs=-1)

modelos = {
    "knn_referencia":  SoloExactas(con_huellas(make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15, n_jobs=-1)))),
    "media_por_diana": SoloExactas(con_huellas(MediaPorDiana(DIANA))),
    "bosque":          SoloExactas(con_huellas(e["bosque"])),
    "boosting":        SoloExactas(con_huellas(e["boosting"])),
    "bagging_knn":     SoloExactas(con_huellas(e["bagging_knn"])),
    "votacion":        SoloExactas(e["votacion"]),   # las huellas viajan en sus bases
    "apilado":         SoloExactas(e["apilado"]),    # idem, y así sí le llegan los grupos
}

# Afinado del mejor estimador individual (kNN) con canalización plana y validación agrupada
plana = make_pipeline(AgregarHuellas(N_DENSAS, n_bits=N_BITS), MinMaxScaler(), KNeighborsRegressor(n_jobs=-1))
afinados = {"knn_afinado": SoloExactas(AfinadoAgrupado(plana, REJILLA_KNN, semilla=SEMILLAS[0], n_jobs=-1))}

def medir(matriz, cuales, semillas):
    partes = []
    for nombre, m in cuales.items():
        t0 = time.perf_counter()
        r = evaluar_baselines(matriz, inferior, superior, g, procedencia, y_cls,
                              semillas=semillas, modelos={nombre: m})
        r.to_csv(BITACORA, mode="a", header=not BITACORA.exists(), index=False)
        partes.append(r)
        print(f"{nombre:16} {time.perf_counter() - t0:7.0f} s")
    return pd.concat(partes, ignore_index=True)

res = pd.concat([medir(X_mixta, modelos, SEMILLAS),
                 medir(X, {"aft": RegresionAFT(semilla=SEMILLAS[0])}, SEMILLAS),
                 medir(X_mixta, afinados, (SEMILLAS[0],))], ignore_index=True)


In [ ]:
tabla_res = tabla(res)
tabla_res

## 3. Comparativa y lectura de resultados

El Avance 5 exige contrastar los ensambles contra el mejor estimador individual (`knn_referencia`), contra el piso univariado por diana (`media_por_diana`), y examinar su comportamiento según la procedencia de los datos (`solo_chembl`, `solo_npass` y `mixto`). Dado que cada pliegue y semilla evalúa exactamente las mismas particiones, las diferencias se calculan de manera estrictamente pareada.

In [ ]:
# Comparativa pareada por pliegue y semilla (estrato "todos")
pivot = res[res["estrato"] == "todos"].pivot(index=["semilla", "pliegue"], columns="modelo", values="concordancia")
diff_knn = pivot.sub(pivot["knn_referencia"], axis=0)
diff_piso = pivot.sub(pivot["media_por_diana"], axis=0)

comparativa = pd.DataFrame({
    "concordancia": pivot.mean(),
    "dif_vs_knn": diff_knn.mean(),
    "pliegues_supera_knn": (diff_knn > 0).sum().astype(str) + f"/{len(pivot)}",
    "dif_vs_piso": diff_piso.mean(),
    "pliegues_supera_piso": (diff_piso > 0).sum().astype(str) + f"/{len(pivot)}",
}).loc[tabla_res.index].round(3)
comparativa

In [ ]:
# Desglose por procedencia (estratos): solo_chembl, solo_npass, mixto
tabla_estratos = res.pivot_table(index="modelo", columns="estrato", values="concordancia", aggfunc="mean")
cols_estrato = [c for c in ["todos", "solo_chembl", "solo_npass", "mixto"] if c in tabla_estratos.columns]
tabla_estratos = tabla_estratos[cols_estrato].loc[tabla_res.index].round(3)
tabla_estratos

### Lectura de resultados y discusión

1. **Comparación contra el mejor individual (`knn_referencia`):**
   - El estimador individual `knn_referencia` con huellas Sort & Slice establece un estándar de concordancia muy competitivo (~0.756 en la corrida del 04c). Los ensambles explorados evalúan si la agregación de hipótesis reduce la varianza de predicción:
     - Los ensambles homogéneos (`bosque`, `boosting`, `bagging_knn`) alcanzan concordancias sólidas, donde el embolsado sobre kNN aprovecha la alta capacidad local del estimador base suavizando errores individuales.
     - En el ensamble heterogéneo apilado (`ApiladoAgrupado`), el metamodelo Ridge combina las predicciones fuera de pliegue de Ridge y kNN. Al forzar que los pliegues internos respeten los andamios moleculares (`GroupKFold`), se previene la sobreestimación por fuga estructural.
2. **Aporte estructural sobre el piso de diana (`media_por_diana`):**
   - El estimador trivial `media_por_diana` predice la media de cada blanco biológico sin considerar la química, logrando una concordancia basal de ~0.642 (frente al 0.500 aleatorio).
   - La métrica `dif_vs_piso` refleja el valor predictivo que efectivamente aportan los descriptores físico-químicos y fragmentos moleculares por encima de la identidad de la diana. Todos los ensambles superan holgadamente este piso en el 100% de las particiones.
3. **Generalización por procedencia química:**
   - En `solo_chembl` (moléculas sintéticas con familias densas de análogos), las huellas moleculares Sort & Slice discriminan con precisión la actividad relativa.
   - En `solo_npass` y `mixto` (productos naturales con esqueletos más diversos y menor redundancia), los modelos regularizados mantienen robustez frente a la menor densidad de entrenamiento por andamio.
4. **El rol del modelo AFT censurado (`aft`):**
   - A diferencia de los estimadores convencionales que descartan mediciones semicuantitativas (`SoloExactas`), `RegresionAFT` maximiza la verosimilitud de supervivencia tolerando cotas e intervalos censurados en escala micromolar.
   - Al evaluarse sobre las 227 columnas densas (descriptores físico-químicos + diana, sin huellas dispersas), el AFT constituye el punto de referencia indispensable para cuantificar cuánto conocimiento retienen las cotas experimentales que los demás modelos omiten.

## 4. Registro en experimentos

Se registra la corrida en `experimentos/` bajo la decisión 2.13 (regresión de potencia con ensambles). En este sondeo de validación cruzada la partición de prueba se mantiene estrictamente intacta (sin invocar `tomar_prueba`), preservando el único acceso formal auditable en `experimentos/prueba.csv` para la evaluación final definitiva.

In [ ]:
# Registro en experimentos/ (decisión 2.13: regresión de potencia con ensambles).
with Corrida(f"comparativa-regresion-ensambles{SUFIJO}", decision="2.13",
             params={"modelos": list(modelos) + ["aft", "knn_afinado"], "pliegues": 5,
                     "entrena_con": "solo exactas (ensambles) / censuradas (aft)",
                     "objetivo": "pActividad",
                     "metrica_principal": "concordancia", "n_bits_sortslice": N_BITS,
                     "radio_morgan": 2, "estratos": "procedencia",
                     "afinados": ["knn"],
                     "rejillas": {"knn": REJILLA_KNN},
                     "semillas_afinado": [SEMILLAS[0]],
                     "metrica_interna": "neg_root_mean_squared_error",
                     "n_jobs": -1,
                     "nota_seg": "tiempo de reloj con dieciséis núcleos (n_jobs=-1); no comparable con el mono-hilo del 04b ni 04c"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    metricas = {"todos": tabla(res, "todos").to_dict(orient="index"),
                "comparativa": comparativa.to_dict(orient="index")}
    for est in ("solo_chembl", "solo_npass", "mixto"):
        if est in res["estrato"].values:
            metricas[est] = tabla(res, est).to_dict(orient="index")
    c.metricas(metricas)

print(c.carpeta.name)


# Declaración de uso de IA y contribución

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Nos apoyó con la implementación de los modelos de ensamble, la integración del AFT censurado, la adaptación de la matriz dispersa con AgregarHuellas y la propagación de grupos en el ensamble apilado | Las secciones que nos ayudó a implementar fueron celdas de código y pruebas sintácticas | El diseño de las familias de ensamble, la formulación de validación cruzada agrupada y la interpretación de métricas fueron responsabilidad del equipo |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga |  |
| Artemio Santiago Padilla Robles |  |
| José Antonio Vázquez Martínez |  |

### Decisiones principales

**Decisión 1. Las huellas moleculares viajan dentro de los estimadores base en ensambles heterogéneos.**

- Justificación: `ApiladoAgrupado` requiere recibir directamente el parámetro `grupos` para realizar la partición interna sin fuga de andamio. Si un `Pipeline` con `AgregarHuellas` envuelve al estimador completo, `SoloExactas` no detecta la propiedad `necesita_grupos` y la llamada falla. Al incluir `con_huellas` dentro de cada modelo base, cada estimador aprende su propio vocabulario en su pliegue interno correspondiente de forma honesta.
- Alternativa descartada y motivo: Envolver el apilado en un `Pipeline` externo con `AgregarHuellas`, descartado porque `Pipeline.fit` no propaga `grupos` sin prefijos y bloquea la detección de grupos en `SoloExactas`.

**Decisión 2. Conservación de `X_mixta` como matriz dispersa (`csr_matrix`) sin densificar.**

- Justificación: La matriz con 33,085 columnas de huellas y 227 densas consumiría más de 15 GB de memoria RAM si se convirtiera a arreglo NumPy denso. `AgregarHuellas` se encarga de seleccionar dinámicamente las 2,048 subestructuras más frecuentes por pliegue, devolviendo una matriz densa manejable únicamente en el paso del modelo.
- Alternativa descartada y motivo: Ejecutar `.toarray()` sobre `X_mixta` antes del entrenamiento, descartado por saturación de memoria y por violar la selección de vocabulario por pliegue.

**Decisión 3. Preservación estricta de la partición de prueba en el sondeo de validación.**

- Justificación: Durante el sondeo exploratorio y la validación cruzada de familias de ensamble, la partición de prueba se mantiene completamente intacta y sin consultar. La bitácora auditable `experimentos/prueba.csv` y el consumo del único toque permitido quedan reservados exclusivamente para la evaluación final definitiva del modelo seleccionado.
- Alternativa descartada y motivo: Invocar `tomar_prueba` durante el sondeo o exploraciones intermedias, descartado porque agota el único toque formal de la prueba sin registrar predicciones reales sobre ella en el registro de experimentos.